# Часть A. Подготовка

Перед запуском выберите GPU: Runtime → Change runtime type → L4, A100 или H100.
Подробный разбор каждой строки — в файле [docs/A_B_explained.md](https://github.com/IvanovskyDev/Machine-Unlearning-in-LLM/blob/main/docs/A_B_explained.md).

**1. Подключаем Google Drive.** Там хранится всё важное: машина Colab очищается после каждой сессии. Colab попросит разрешение — согласитесь.

In [ ]:
from google.colab import drive   # модуль Colab для работы с Google Drive

drive.mount("/content/drive")    # подключить Drive: его файлы появятся в папке /content/drive

**2. Создаём папки проекта.** На Drive — для чекпоинтов и результатов, на диске машины — для моделей и кэша (их можно скачать заново).

In [ ]:
import os   # модуль для работы с папками и переменными окружения

DRIVE = "/content/drive/MyDrive/unlearning_data"  # папка проекта на Google Drive (постоянная)
FAST = "/content/fast"                             # папка на диске машины (очищается после сессии)

# os.makedirs создаёт папку; exist_ok=True — если папка уже есть, ничего не делать
os.makedirs(DRIVE + "/saves", exist_ok=True)        # чекпоинты моделей
os.makedirs(DRIVE + "/results_raw", exist_ok=True)  # сырые результаты атак
os.makedirs(FAST + "/hf_home", exist_ok=True)       # кэш Hugging Face
os.makedirs(FAST + "/models", exist_ok=True)        # скачанные модели

**3. Задаём переменные окружения.** По ним программы находят папки из шага 2. Должны напечататься два пути.

In [ ]:
# os.environ — переменные окружения: их видят все программы, которые запускает блокнот
os.environ["BIG"] = DRIVE                       # «большой диск» из плана
os.environ["HF_HOME"] = FAST + "/hf_home"       # куда библиотеки Hugging Face складывают скачанное
os.environ["MODELS"] = FAST + "/models"         # куда будем скачивать модели (часть C)
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # меньше лишних предупреждений токенизаторов
os.environ["PYTHONUNBUFFERED"] = "1"            # вывод программ сразу попадает в лог

# ! в начале строки — команда терминала; echo печатает значение переменной
!echo $HF_HOME
!echo $MODELS

**4. Подключаем токен Hugging Face.** Должно напечататься ваше имя на Hugging Face. Токен добавляется заранее, один раз: huggingface.co → Settings → Access Tokens → создать токен типа Read; в Colab слева 🔑 Secrets → добавить `HF_TOKEN` и включить доступ для блокнота.

In [ ]:
from google.colab import userdata    # доступ к секретам Colab (🔑 на левой панели)
from huggingface_hub import whoami   # функция «кто я» на Hugging Face

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # взять токен из секрета HF_TOKEN — в коде его нет
print("Hugging Face:", whoami()["name"])            # спросить у Hugging Face имя владельца токена

**5. Смотрим GPU.** В таблице должна быть L4, A100 или H100 (T4 для обучения не подходит), а справа вверху — CUDA Version 12 или выше.

In [ ]:
# таблица о GPU: модель, память, версия драйвера и CUDA, запущенные на GPU процессы
!nvidia-smi

**6. Записываем характеристики машины в журнал** (`journal.md` на Drive, пригодится для главы 3 диплома). Выполняйте один раз для каждой новой модели GPU.

In [ ]:
import shutil                    # здесь — чтобы узнать свободное место на диске
from datetime import datetime    # текущие дата и время
from zoneinfo import ZoneInfo    # часовые пояса

import psutil                    # сведения о машине: здесь — объём оперативной памяти

# x = !команда — выполнить команду терминала и сохранить её вывод в x (список строк)
gpu = !nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader
cuda = !nvidia-smi | grep -o "CUDA Version: [0-9.]*"

ram = round(psutil.virtual_memory().total / 1e9)          # ОЗУ в гигабайтах (1e9 — миллиард байт)
disk = round(shutil.disk_usage("/content").free / 1e9)    # свободное место на диске машины в гигабайтах
today = datetime.now(ZoneInfo("Europe/Moscow")).date()    # сегодняшняя дата по Москве (часы Colab идут по UTC)

# f"""...""" — многострочный текст; вместо {...} подставляются значения переменных
text = f"""
## {today} — Проверка машины (Colab)
- GPU (имя, память, compute capability, драйвер): {gpu[0]}
- {cuda[0]}
- CPU: {os.cpu_count()} ядер, ОЗУ: {ram} ГБ, свободно на диске машины: {disk} ГБ
- Большой диск: {DRIVE}
"""

# открыть журнал на дописывание ("a") и добавить запись в конец файла
with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:
    f.write(text)

print(text)   # показать запись на экране